# Validate 1-MIN → 1-HOUR Aggregation in NautilusTrader

Proves NautilusTrader's `TimeBarAggregator` correctly produces `1-HOUR-{BID,ASK}-INTERNAL` bars from `1-MINUTE-{BID,ASK}-EXTERNAL` source data — round-tripping the source through a CSV.

## Pipeline
```
csv/raw_engine_data_1min_2024_04.csv           (1-MIN EXTERNAL, EURUSD.FOREX_MS, April 2024)
         │
         ▼  pandas → Bar(Price.from_str, Quantity.from_str)
engine.add_data(all_bars)
         │
         ▼  engine.run()
DataEngine ─dispatches 1-MIN-EXT bars─→ BarSniffer.on_bar         (stream='sniffer_engine')
         │
         ▼  TimeBarAggregator (one per INTERNAL@…-EXTERNAL subscription)
BarSniffer.on_bar                                                  (stream='sniffer_strategy')
```

## Sections
1. Imports & config
2. Load raw 1-MIN CSV → `Bar` objects
3. Engine + venue + instrument setup
4. DataEngine sniffer wiring (rationale)
5. `BarSniffer` definition (1-HOUR-INTERNAL subscriber + EXTERNAL recorder)
6. Run backtest
7. Validation report (four explicit assertions)
8. Write three output CSVs to `./csv/`
9. Summary table

## Deliverables (all overwrite-in-place in `./csv/`)
- `csv/raw_engine_data_1min_2024_04.csv` — what `engine.add_data(...)` actually staged (re-serialized from the `Bar` objects we built)
- `csv/sniffer_engine_1min_2024_04.csv` — what the `DataEngine` dispatched (1-MIN-EXTERNAL)
- `csv/sniffer_strategy_1h_2024_04.csv` — what `on_bar` received (1-HOUR-INTERNAL)

**`./csv/` already exists** and contains other files (5-min, 15-min sniffers from sibling notebooks). This notebook only overwrites the three deliverable filenames above; everything else in `./csv/` is left untouched.

All three share the same 12-column schema:
```
stream,price_type,bar_type,ts_event_ns,ts_init_ns,open,high,low,close,volume,ts_event,ts_init
```

## 1. Imports & config

In [1]:
from importlib.metadata import version as _pkg_version
print(f"nautilus_trader version: {_pkg_version('nautilus_trader')}")

nautilus_trader version: 1.224.0


In [2]:
import csv
import random
import sys
from decimal import Decimal
from pathlib import Path

import pandas as pd


def _find_project_root() -> Path:
    marker = Path("core") / "instrument_factory.py"
    def _walk_up(p: Path) -> Path | None:
        for parent in (p, *p.parents):
            if (parent / marker).is_file():
                return parent
        return None
    found = _walk_up(Path.cwd())
    if found:
        return found
    for var in ("__vsc_ipynb_file__", "__session__", "__file__"):
        nb = globals().get(var)
        if nb:
            found = _walk_up(Path(nb).resolve().parent)
            if found:
                return found
    for child in Path.cwd().iterdir():
        if child.is_dir() and (child / marker).is_file():
            return child
    raise RuntimeError(f"could not find project root (no `{marker}`) from cwd={Path.cwd()}")


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CSV_DIR = PROJECT_ROOT / "csv"
assert CSV_DIR.is_dir(), f"expected pre-existing {CSV_DIR}"

INSTRUMENT_ID_STR     = "EURUSD.FOREX_MS"
RAW_1MIN_CSV          = CSV_DIR / "raw_engine_data_1min_2024_04.csv"
OUT_RAW_ENGINE        = CSV_DIR / "raw_engine_data_1min_2024_04.csv"      # overwrites in place
OUT_SNIFFER_ENGINE    = CSV_DIR / "sniffer_engine_1min_2024_04.csv"       # overwrites in place
OUT_SNIFFER_STRATEGY  = CSV_DIR / "sniffer_strategy_1h_2024_04.csv"       # new file (1h convention)

assert RAW_1MIN_CSV.exists(), (
    f"missing input {RAW_1MIN_CSV}; run ipynb/engine_dispatch_verification.ipynb "
    f"first to populate csv/"
)
print(f"project root: {PROJECT_ROOT}")
print(f"csv dir     : {CSV_DIR}")
print(f"input       : {RAW_1MIN_CSV.name}  ({RAW_1MIN_CSV.stat().st_size/1024:,.1f} KB)")

project root: c:\Users\HP\OneDrive\Desktop\m-cube_version1
csv dir     : c:\Users\HP\OneDrive\Desktop\m-cube_version1\csv
input       : raw_engine_data_1min_2024_04.csv  (11,320.3 KB)


In [3]:
from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.cache.config import CacheConfig
from nautilus_trader.config import BacktestEngineConfig, LoggingConfig, RiskEngineConfig, StrategyConfig
from nautilus_trader.model import TraderId
from nautilus_trader.model.currencies import USD
from nautilus_trader.model.data import Bar, BarType
from nautilus_trader.model.enums import AccountType, AggregationSource, OmsType
from nautilus_trader.model.identifiers import InstrumentId
from nautilus_trader.model.objects import Money, Price, Quantity
from nautilus_trader.trading.strategy import Strategy

from core.instrument_factory import create_instrument


# Build instrument up front so its precisions are available to the row→Bar parser below.
instrument = create_instrument("EUR", "USD", venue="FOREX_MS")
PRICE_PREC = instrument.price_precision   # EURUSD: 5
SIZE_PREC  = instrument.size_precision    # EURUSD: 0 (whole-unit volume)
print(f"instrument: {instrument.id}  price_precision={PRICE_PREC}  size_precision={SIZE_PREC}")


# Shared 12-column schema — used for both reading and writing.
CSV_FIELDS = [
    "stream", "price_type", "bar_type",
    "ts_event_ns", "ts_init_ns",
    "open", "high", "low", "close", "volume",
    "ts_event", "ts_init",
]


def bar_to_row(stream: str, bar: Bar) -> dict:
    bt = bar.bar_type
    return {
        "stream":      stream,
        "price_type":  bt.spec.price_type.name,
        "bar_type":    str(bt),
        "ts_event_ns": int(bar.ts_event),
        "ts_init_ns":  int(bar.ts_init),
        "open":        float(bar.open),
        "high":        float(bar.high),
        "low":         float(bar.low),
        "close":       float(bar.close),
        "volume":      float(bar.volume),
        "ts_event":    pd.Timestamp(bar.ts_event, unit="ns", tz="UTC").isoformat(),
        "ts_init":     pd.Timestamp(bar.ts_init,  unit="ns", tz="UTC").isoformat(),
    }


def write_rows(path, rows):
    with open(path, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=CSV_FIELDS)
        w.writeheader()
        w.writerows(rows)

instrument: EURUSD.FOREX_MS  price_precision=5  size_precision=0


## 2. Load raw 1-MIN CSV → `Bar` objects

Reads OHLC columns as **strings** to preserve exact source representation. Each price is then reformatted to `instrument.price_precision` digits and handed to `Price.from_str` — this restores trailing zeros that the prior CSV write dropped (e.g. `1.0793` in the CSV reconstructs as `Price(1.07930, 5)` to match `instrument.price_precision=5`). Volumes are integer-valued in the source; we truncate the `.0` and use `Quantity.from_str` for precision=0.

The raw CSV is read into memory **before** we get to section 8 (which overwrites the same file). Once the `Bar` list is built, the on-disk file can be replaced without affecting downstream work.

In [4]:
df_1min = pd.read_csv(
    RAW_1MIN_CSV,
    dtype={
        "stream": str, "price_type": str, "bar_type": str,
        "open": str, "high": str, "low": str, "close": str, "volume": str,
        "ts_event": str, "ts_init": str,
    },
)
# ts_event_ns / ts_init_ns are still parsed as int (default numeric dtype).
print(f"loaded {len(df_1min):,} rows from {RAW_1MIN_CSV.name}")
print(f"price_type counts: {df_1min['price_type'].value_counts().to_dict()}")
print(f"bar_types in source: {sorted(df_1min['bar_type'].unique())}")

bt_1m_bid = BarType.from_str(f"{INSTRUMENT_ID_STR}-1-MINUTE-BID-EXTERNAL")
bt_1m_ask = BarType.from_str(f"{INSTRUMENT_ID_STR}-1-MINUTE-ASK-EXTERNAL")


def _to_price(s: str) -> Price:
    # Force the instrument's price_precision so trailing zeros survive the CSV round-trip.
    return Price.from_str(f"{float(s):.{PRICE_PREC}f}")


def _to_quantity(s: str) -> Quantity:
    # Source CSV writes whole volumes as `144.0`; coerce to integer string for precision=0.
    if SIZE_PREC == 0:
        return Quantity.from_str(str(int(float(s))))
    return Quantity.from_str(f"{float(s):.{SIZE_PREC}f}")


def row_to_bar(row, bar_type: BarType) -> Bar:
    return Bar(
        bar_type=bar_type,
        open=_to_price(row["open"]),
        high=_to_price(row["high"]),
        low=_to_price(row["low"]),
        close=_to_price(row["close"]),
        volume=_to_quantity(row["volume"]),
        ts_event=int(row["ts_event_ns"]),
        ts_init=int(row["ts_init_ns"]),
    )


bid_rows = df_1min[df_1min["price_type"] == "BID"].to_dict(orient="records")
ask_rows = df_1min[df_1min["price_type"] == "ASK"].to_dict(orient="records")

bars_bid = [row_to_bar(r, bt_1m_bid) for r in bid_rows]
bars_ask = [row_to_bar(r, bt_1m_ask) for r in ask_rows]
all_bars = bars_bid + bars_ask

n_input_bid = len(bid_rows)
n_input_ask = len(ask_rows)
print(f"constructed: BID={n_input_bid:,}  ASK={n_input_ask:,}  total={len(all_bars):,}")
print(f"sample BID bar: open.precision={bars_bid[0].open.precision} "
      f"volume.precision={bars_bid[0].volume.precision}")

loaded 63,360 rows from raw_engine_data_1min_2024_04.csv
price_type counts: {'BID': 31680, 'ASK': 31680}
bar_types in source: ['EURUSD.FOREX_MS-1-MINUTE-ASK-EXTERNAL', 'EURUSD.FOREX_MS-1-MINUTE-BID-EXTERNAL']
constructed: BID=31,680  ASK=31,680  total=63,360
sample BID bar: open.precision=5 volume.precision=0


## 3. Engine + venue + instrument setup

Engine config mirrors [core/backtest_runner.py](../core/backtest_runner.py) (`_run_single_slot`): bypass logging + risk, `run_analysis=False`. `CacheConfig(bar_capacity=200_000)` defeats the 10,000-bar `deque` truncation that would silently drop most of April 2024 from `engine.cache.bars()`.

In [5]:
engine = BacktestEngine(config=BacktestEngineConfig(
    trader_id=TraderId("AGG-1H-VALIDATE"),
    logging=LoggingConfig(bypass_logging=True),
    risk_engine=RiskEngineConfig(bypass=True),
    run_analysis=False,
    cache=CacheConfig(bar_capacity=200_000, drop_instruments_on_reset=False),
))
engine.add_venue(
    venue=instrument.id.venue,
    oms_type=OmsType.NETTING,
    account_type=AccountType.MARGIN,
    starting_balances=[Money(1_000_000, USD)],
    base_currency=USD,
    default_leverage=Decimal(1),
)
engine.add_instrument(instrument)
engine.add_data(all_bars)

# Snapshot engine.data BEFORE run() — it's a property returning a copy of the
# internal _data list. Needed for assertion 1 and the raw_engine output CSV.
engine_data_snapshot = list(engine.data)
print(f"engine.data snapshot: {len(engine_data_snapshot):,} items")

engine.data snapshot: 63,360 items


## 4. DataEngine sniffer wiring (rationale)

The spec offered two implementation choices for capturing what the `DataEngine` processes:
1. Monkey-patch `engine.kernel.data_engine._handle_bar`.
2. Read `engine.cache.bars(bar_type)` after the run.

We use a **third, equivalent approach**: a single `BarSniffer` strategy that subscribes to both the 1-MIN-EXTERNAL feed and the 1-HOUR-INTERNAL composite, then routes incoming bars to one of two lists based on `bar.bar_type.aggregation_source`. This is the most version-stable option (no `_handle_bar` private API, no post-run cache iteration that's vulnerable to capacity truncation) and was verified end-to-end in `ipynb/engine_dispatch_verification.ipynb`.

The semantic contract is identical: every bar the `DataEngine` dispatches to **any subscriber** must flow through `on_bar` exactly once per subscription. Since the strategy subscribes to the 1-MIN-EXTERNAL types directly, `on_bar(bar)` calls with `aggregation_source==EXTERNAL` are the same bars the `DataEngine` processed.

## 5. `BarSniffer` definition

Subscribes to four `BarType`s. The 1-hour subscriptions use the explicit composite form (`…-1-HOUR-{BID,ASK}-INTERNAL@1-MINUTE-EXTERNAL`) which tells Nautilus to source the aggregator from the EXTERNAL feed already in the engine. The emitted aggregated bar's `str(bar.bar_type)` returns the short head form (`…-1-HOUR-{BID,ASK}-INTERNAL`), which is what lands in the output CSV's `bar_type` column.

In [6]:
class BarSnifferConfig(StrategyConfig, frozen=True):
    instrument_id: InstrumentId
    bar_types: tuple[BarType, ...]


class BarSniffer(Strategy):
    """Bucket every on_bar into engine_rows (EXTERNAL) or strategy_rows (INTERNAL)."""

    def __init__(self, config: BarSnifferConfig) -> None:
        super().__init__(config)
        self.engine_rows:   list[dict] = []
        self.strategy_rows: list[dict] = []

    def on_start(self) -> None:
        # EXTERNAL before INTERNAL — aggregator wires to source feed cleanly.
        for bt in sorted(self.config.bar_types, key=lambda b: b.aggregation_source.value):
            self.subscribe_bars(bt)

    def on_bar(self, bar: Bar) -> None:
        if bar.bar_type.aggregation_source == AggregationSource.EXTERNAL:
            self.engine_rows.append(bar_to_row("sniffer_engine", bar))
        else:
            self.strategy_rows.append(bar_to_row("sniffer_strategy", bar))

## 6. Run backtest

In [7]:
bt_1h_bid_int = BarType.from_str(f"{INSTRUMENT_ID_STR}-1-HOUR-BID-INTERNAL@1-MINUTE-EXTERNAL")
bt_1h_ask_int = BarType.from_str(f"{INSTRUMENT_ID_STR}-1-HOUR-ASK-INTERNAL@1-MINUTE-EXTERNAL")

sniffer = BarSniffer(BarSnifferConfig(
    instrument_id=instrument.id,
    bar_types=(bt_1m_bid, bt_1m_ask, bt_1h_bid_int, bt_1h_ask_int),
))
engine.add_strategy(sniffer)
engine.run()

print(f"engine_rows  (1-MIN EXT, sniffer_engine):   {len(sniffer.engine_rows):,}")
print(f"strategy_rows(1-HOUR INT, sniffer_strategy): {len(sniffer.strategy_rows):,}")

engine_rows  (1-MIN EXT, sniffer_engine):   63,360
strategy_rows(1-HOUR INT, sniffer_strategy): 1,440


## 7. Validation report

Four explicit assertions. Each prints `PASS` / `FAIL` first; an `AssertionError` halts execution if any fail.

- **A1** — `engine.data` contains the right bar types & counts.
- **A2** — Sniffer's EXTERNAL list matches the source CSV row-for-row (OHLCV + ts_event_ns), using `Decimal` for exact compare.
- **A3** — Strategy received 1-HOUR-INTERNAL bars on both sides, count ≈ `floor(num_1min / 60)` with boundary tolerance.
- **A4** — Aggregation math: for 3 random non-empty windows per side, recompute OHLCV from the 1-min source and compare via `Decimal` (no float drift).

In [8]:
df_eng    = pd.DataFrame(sniffer.engine_rows)
df_strat  = pd.DataFrame(sniffer.strategy_rows)

BT_1M_BID = f"{INSTRUMENT_ID_STR}-1-MINUTE-BID-EXTERNAL"
BT_1M_ASK = f"{INSTRUMENT_ID_STR}-1-MINUTE-ASK-EXTERNAL"
BT_1H_BID = f"{INSTRUMENT_ID_STR}-1-HOUR-BID-INTERNAL"
BT_1H_ASK = f"{INSTRUMENT_ID_STR}-1-HOUR-ASK-INTERNAL"

# === A1 — engine.data correctness ===
bars_in_data = [d for d in engine_data_snapshot if isinstance(d, Bar)]
bid_in_data = [b for b in bars_in_data if str(b.bar_type) == BT_1M_BID]
ask_in_data = [b for b in bars_in_data if str(b.bar_type) == BT_1M_ASK]
assert len(bid_in_data) == n_input_bid, f"A1 BID mismatch: {len(bid_in_data)} vs {n_input_bid}"
assert len(ask_in_data) == n_input_ask, f"A1 ASK mismatch: {len(ask_in_data)} vs {n_input_ask}"
assert len(bid_in_data) + len(ask_in_data) == len(bars_in_data), "A1: stray bar types in engine.data"
print(f"PASS A1: engine.data has {len(bid_in_data):,} BID + {len(ask_in_data):,} ASK 1-MIN-EXTERNAL bars")

# === A2 — DataEngine saw the same 1-MIN-EXTERNAL bars ===
src = df_1min[["bar_type", "ts_event_ns", "open", "high", "low", "close", "volume"]].copy()
snf = df_eng[["bar_type", "ts_event_ns", "open", "high", "low", "close", "volume"]].copy()
src["ts_event_ns"] = src["ts_event_ns"].astype(int)
snf["ts_event_ns"] = snf["ts_event_ns"].astype(int)
for col in ("open", "high", "low", "close", "volume"):
    src[col] = src[col].astype(str).map(lambda s: Decimal(s))
    snf[col] = snf[col].astype(str).map(lambda s: Decimal(s))
src_sorted = src.sort_values(["bar_type", "ts_event_ns"]).reset_index(drop=True)
snf_sorted = snf.sort_values(["bar_type", "ts_event_ns"]).reset_index(drop=True)
assert len(src_sorted) == len(snf_sorted), f"A2 row count: {len(src_sorted)} vs {len(snf_sorted)}"
diff_mask = (src_sorted != snf_sorted).any(axis=1)
n_diff = int(diff_mask.sum())
if n_diff:
    print("A2 first 3 diffs (source vs sniffer):")
    print(src_sorted[diff_mask].head(3))
    print("---")
    print(snf_sorted[diff_mask].head(3))
assert n_diff == 0, f"A2: {n_diff} differing rows between source CSV and sniffer_engine"
print(f"PASS A2: sniffer_engine row-for-row identical to source CSV ({len(snf_sorted):,} rows)")

# === A3 — Strategy received 1-HOUR-INTERNAL on both sides, count in expected band ===
strat_bid = df_strat[df_strat["bar_type"] == BT_1H_BID]
strat_ask = df_strat[df_strat["bar_type"] == BT_1H_ASK]
assert len(strat_bid) > 0, "A3: no 1-HOUR BID INTERNAL bars received"
assert len(strat_ask) > 0, "A3: no 1-HOUR ASK INTERNAL bars received"
expected = n_input_bid // 60           # data-driven lower bound
upper    = 30 * 24 + 5                  # clock-driven upper: 30 days × 24 hours + slack
assert expected - 5 <= len(strat_bid) <= upper, f"A3 BID {len(strat_bid)} outside [{expected-5}, {upper}]"
assert expected - 5 <= len(strat_ask) <= upper, f"A3 ASK {len(strat_ask)} outside [{expected-5}, {upper}]"
print(f"PASS A3: 1-HOUR INTERNAL bars received — BID={len(strat_bid)}  ASK={len(strat_ask)}  "
      f"(data-driven lower≈{expected}, clock-driven upper={upper})")

# === A4 — Aggregation math (Decimal-exact, 3 random non-empty windows per side) ===
NS_PER_MIN = 60 * 1_000_000_000
WINDOW_NS  = 60 * NS_PER_MIN

src_1min = df_1min.copy()
src_1min["ts_event_ns"] = src_1min["ts_event_ns"].astype(int)
for col in ("open", "high", "low", "close", "volume"):
    src_1min[col] = src_1min[col].map(lambda s: Decimal(str(s)))

rng = random.Random(42)
checks = 0
for side, strat_side, src_bar_type in [
    ("BID", strat_bid, BT_1M_BID),
    ("ASK", strat_ask, BT_1M_ASK),
]:
    src_for_side = src_1min[src_1min["bar_type"] == src_bar_type].sort_values("ts_event_ns")
    src_ts_set = set(src_for_side["ts_event_ns"].tolist())
    candidates = []
    for _, row in strat_side.iterrows():
        t_close = int(row["ts_event_ns"])
        lo = t_close - WINDOW_NS + NS_PER_MIN  # 1-min bars in (t_close-60min, t_close]
        if any((lo <= ts <= t_close) for ts in src_ts_set):
            candidates.append(t_close)
    sample = rng.sample(candidates, k=3)
    for t_close in sample:
        lo = t_close - WINDOW_NS + NS_PER_MIN
        window_1min = src_for_side[(src_for_side["ts_event_ns"] >= lo) &
                                   (src_for_side["ts_event_ns"] <= t_close)]
        assert len(window_1min) > 0, f"empty window picked at t={t_close} (sampler bug)"
        manual = {
            "open":   window_1min.iloc[0]["open"],
            "high":   max(window_1min["high"]),
            "low":    min(window_1min["low"]),
            "close":  window_1min.iloc[-1]["close"],
            "volume": sum(window_1min["volume"]),
        }
        row1h = strat_side[strat_side["ts_event_ns"] == t_close].iloc[0]
        actual = {k: Decimal(str(row1h[k])) for k in ("open", "high", "low", "close", "volume")}
        for k in manual:
            assert manual[k] == actual[k], (
                f"A4 {side} t={t_close} {k}: manual={manual[k]} actual={actual[k]} "
                f"(window had {len(window_1min)} 1-min bars)"
            )
        checks += 1
print(f"PASS A4: aggregation math exact for {checks} sampled 1-hour windows (Decimal compare)")

engine.dispose()

PASS A1: engine.data has 31,680 BID + 31,680 ASK 1-MIN-EXTERNAL bars
PASS A2: sniffer_engine row-for-row identical to source CSV (63,360 rows)
PASS A3: 1-HOUR INTERNAL bars received — BID=720  ASK=720  (data-driven lower≈528, clock-driven upper=725)
PASS A4: aggregation math exact for 6 sampled 1-hour windows (Decimal compare)


## 8. Write three output CSVs to `./csv/`

All three written with the shared 12-column schema and identical formatting. Writes overwrite the existing `raw_engine_data_1min_2024_04.csv` and `sniffer_engine_1min_2024_04.csv` in place (both regenerated from the round-tripped Bar objects + DataEngine dispatch); `sniffer_strategy_1h_2024_04.csv` is a new file. Other contents of `./csv/` (5-min, 15-min sniffers from sibling notebooks) are not touched.

In [9]:
# (a) raw_engine — re-serialized from the Bar objects we built (what engine.add_data staged).
raw_rows_out = [
    bar_to_row("raw_engine", d)
    for d in engine_data_snapshot
    if isinstance(d, Bar) and d.bar_type.aggregation_source == AggregationSource.EXTERNAL
]
write_rows(OUT_RAW_ENGINE, raw_rows_out)

# (b) sniffer_engine — captured by BarSniffer (EXTERNAL bucket).
write_rows(OUT_SNIFFER_ENGINE, sniffer.engine_rows)

# (c) sniffer_strategy — captured by BarSniffer (INTERNAL bucket).
write_rows(OUT_SNIFFER_STRATEGY, sniffer.strategy_rows)

for p in (OUT_RAW_ENGINE, OUT_SNIFFER_ENGINE, OUT_SNIFFER_STRATEGY):
    size_kb = p.stat().st_size / 1024
    print(f"wrote {p.relative_to(PROJECT_ROOT)}  ({size_kb:,.1f} KB)")

wrote csv\raw_engine_data_1min_2024_04.csv  (11,320.3 KB)
wrote csv\sniffer_engine_1min_2024_04.csv  (11,567.8 KB)
wrote csv\sniffer_strategy_1h_2024_04.csv  (264.5 KB)


## 9. Summary table — counts per stream / price_type / bar_type

In [10]:
summary_parts = []
for path in (OUT_RAW_ENGINE, OUT_SNIFFER_ENGINE, OUT_SNIFFER_STRATEGY):
    df = pd.read_csv(path)
    summary_parts.append(
        df.groupby(["stream", "price_type", "bar_type"])
          .size().reset_index(name="rows")
    )
summary = pd.concat(summary_parts, ignore_index=True)
print(summary.to_string(index=False))

          stream price_type                              bar_type  rows
      raw_engine        ASK EURUSD.FOREX_MS-1-MINUTE-ASK-EXTERNAL 31680
      raw_engine        BID EURUSD.FOREX_MS-1-MINUTE-BID-EXTERNAL 31680
  sniffer_engine        ASK EURUSD.FOREX_MS-1-MINUTE-ASK-EXTERNAL 31680
  sniffer_engine        BID EURUSD.FOREX_MS-1-MINUTE-BID-EXTERNAL 31680
sniffer_strategy        ASK   EURUSD.FOREX_MS-1-HOUR-ASK-INTERNAL   720
sniffer_strategy        BID   EURUSD.FOREX_MS-1-HOUR-BID-INTERNAL   720
